In [6]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(r"D:\PROJECTS\Customer_Review_Intelligence\.env")

api_key = os.getenv("OPENAI_API_KEY")

print("API key found:", bool(api_key))

client = OpenAI(api_key=api_key)

print("Client ready")

API key found: True
Client ready


In [7]:
response = client.responses.create(
    model="gpt-6-luna",
    input="Explain customer review analysis in one simple sentence."
)

print(response.output_text)

Customer review analysis is the process of examining customer feedback to understand what people like, dislike, and want improved.


In [8]:
#clustered data 
import pandas as pd

df = pd.read_csv(
    "../data/processed/harmony_reviews_with_clusters.csv"
)

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (1604, 34)
['id', 'asins', 'brand', 'categories', 'colors', 'dateAdded', 'dateUpdated', 'dimension', 'ean', 'imageURLs', 'keys', 'manufacturer', 'manufacturerNumber', 'name', 'primaryCategories', 'reviews.date', 'reviews.dateSeen', 'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title', 'reviews.username', 'sourceURLs', 'upc', 'weight', 'original_text', 'cleaned_text', 'cleaned_length', 'cluster', 'cluster_v3', 'semantic_cluster', 'semantic_cluster_final']


In [9]:
#Define your categories
categories = [
    "Setup and Programming",
    "Touchscreen and Interface",
    "Device Compatibility",
    "Software and App",
    "TV and Media Control",
    "Design and Ergonomics",
    "Battery and Charging",
    "General Product Experience"
]

print(categories)

['Setup and Programming', 'Touchscreen and Interface', 'Device Compatibility', 'Software and App', 'TV and Media Control', 'Design and Ergonomics', 'Battery and Charging', 'General Product Experience']


In [10]:
#Create the 50-review evaluation set
evaluation_df = df.sample(
    50,
    random_state=42
).copy()

evaluation_df = evaluation_df[
    ["original_text"]
].copy()

evaluation_df["true_category"] = ""

evaluation_df.to_csv(
    "../outputs/manual_labeling_50.csv",
    index=False
)

print("Created:", "../outputs/manual_labeling_50.csv")
print("Reviews:", len(evaluation_df))

Created: ../outputs/manual_labeling_50.csv
Reviews: 50


In [11]:
import pandas as pd

evaluation_df = pd.read_csv(
    "../outputs/manual_labeling_50_completed.csv"
)

print(evaluation_df.shape)
print(evaluation_df["true_category"].value_counts())

(50, 2)
true_category
General Product Experience    14
Setup and Programming         11
TV and Media Control           7
Device Compatibility           6
Battery and Charging           6
Design and Ergonomics          2
Touchscreen and Interface      2
Software and App               2
Name: count, dtype: int64


In [12]:
print("Missing labels:", evaluation_df["true_category"].isna().sum())
print("Blank labels:", (evaluation_df["true_category"].str.strip() == "").sum())

Missing labels: 0
Blank labels: 0


In [13]:
#Zero-shot classifier
def zero_shot_classify(review):

    prompt = f"""
You are a customer review classification system.

Classify this customer review into exactly ONE category.

Categories:
{categories}

Rules:
- Choose exactly one category.
- Return only the category name.
- Do not explain your answer.
- Use the category name exactly as provided.

Review:
{review}
"""

    response = client.responses.create(
        model="gpt-6-luna",
        input=prompt
    )

    return response.output_text.strip()

In [14]:
print(zero_shot_classify(
    evaluation_df["original_text"].iloc[0]
))

General Product Experience


In [15]:
evaluation_df["zero_shot"] = evaluation_df[
    "original_text"
].apply(zero_shot_classify)

print("Zero-shot completed.")

Zero-shot completed.


In [16]:
evaluation_df.to_csv(
    "../outputs/zero_shot_results.csv",
    index=False
)

print("Saved zero-shot results.")

Saved zero-shot results.


In [17]:
#Create balanced few-shot examples
few_shot_examples = (
    evaluation_df
    .groupby("true_category", group_keys=False)
    .head(4)
)

print(
    few_shot_examples["true_category"].value_counts()
)

true_category
Device Compatibility          4
Setup and Programming         4
General Product Experience    4
TV and Media Control          4
Battery and Charging          4
Design and Ergonomics         2
Touchscreen and Interface     2
Software and App              2
Name: count, dtype: int64


In [18]:
examples_text = "\n\n".join(
    f"Review: {row['original_text']}\n"
    f"Category: {row['true_category']}"
    for _, row in few_shot_examples.iterrows()
)

In [19]:
#Create held-out test set
test_df = evaluation_df.drop(
    few_shot_examples.index
).copy()

print("Few-shot examples:", len(few_shot_examples))
print("Test reviews:", len(test_df))

Few-shot examples: 26
Test reviews: 24


In [20]:
#Few-shot classifier
def few_shot_classify(review):

    prompt = f"""
You are a customer review classification system.

Classify the review into exactly ONE category.

Categories:
{categories}

Here are examples of correctly classified reviews:

{examples_text}

Rules:
- Choose exactly one category.
- Follow the examples.
- Use exactly one category name from the list.
- Do not explain your answer.
- Return only the category name.

Review:
{review}
"""

    response = client.responses.create(
        model="gpt-6-luna",
        input=prompt
    )

    return response.output_text.strip()

In [21]:
test_df["few_shot"] = test_df[
    "original_text"
].apply(few_shot_classify)

print("Few-shot completed.")

Few-shot completed.


In [22]:
#Compare Zero-shot vs Few-shot
from sklearn.metrics import classification_report

print("=" * 60)
print("ZERO-SHOT")
print("=" * 60)

print(
    classification_report(
        test_df["true_category"],
        test_df["zero_shot"],
        zero_division=0
    )
)

print("=" * 60)
print("FEW-SHOT")
print("=" * 60)

print(
    classification_report(
        test_df["true_category"],
        test_df["few_shot"],
        zero_division=0
    )
)

ZERO-SHOT
                            precision    recall  f1-score   support

      Battery and Charging       0.00      0.00      0.00         2
     Design and Ergonomics       0.00      0.00      0.00         0
      Device Compatibility       0.00      0.00      0.00         2
General Product Experience       0.40      0.40      0.40        10
     Setup and Programming       0.38      0.43      0.40         7
      TV and Media Control       0.50      0.33      0.40         3

                  accuracy                           0.33        24
                 macro avg       0.21      0.19      0.20        24
              weighted avg       0.34      0.33      0.33        24

FEW-SHOT
                            precision    recall  f1-score   support

      Battery and Charging       0.00      0.00      0.00         2
     Design and Ergonomics       0.00      0.00      0.00         0
      Device Compatibility       0.00      0.00      0.00         2
General Product Experienc

In [23]:
#Accuracy
from sklearn.metrics import accuracy_score

zero_acc = accuracy_score(
    test_df["true_category"],
    test_df["zero_shot"]
)

few_acc = accuracy_score(
    test_df["true_category"],
    test_df["few_shot"]
)

print("Zero-shot accuracy:", round(zero_acc, 3))
print("Few-shot accuracy:", round(few_acc, 3))

Zero-shot accuracy: 0.333
Few-shot accuracy: 0.208


In [24]:
#save
test_df.to_csv(
    "../outputs/zero_vs_few_shot_results.csv",
    index=False
)

print("Saved:", "../outputs/zero_vs_few_shot_results.csv")

Saved: ../outputs/zero_vs_few_shot_results.csv
